# 07 · Bring your own dataset

Any dataset in the ContextXR format can be validated and evaluated (format: `docs/data_format.md`). It is a directory with two files:

- `facet_db.json`: apps, facets with their capabilities, and the facet-graph edges;
- `events.csv`: one row per element a participant placed, with its context and the capabilities it exposes (target facets are derived from these).

Below we write a tiny two-app dataset, catch a mistake with the validator, then evaluate.

In [ ]:
import json, tempfile
from pathlib import Path

root = Path(tempfile.mkdtemp())
db = {
    "format": "contextxr.facet_db", "format_version": "1.0", "name": "my-study",
    "apps": [
        {"app_id": "music", "app_name": "Music", "entry_facet_id": "music.open_section",
         "capabilities": ["open_now_playing", "view_title", "pause"]},
        {"app_id": "recipes", "app_name": "Recipes", "entry_facet_id": "recipes.open_section",
         "capabilities": ["open_view_recipe", "view_ingredients", "open_cook_recipe", "view_step"]},
    ],
    "facets": [
        {"facet_id": "music.open_section", "app_id": "music", "name": "Open Section", "capabilities": ["open_now_playing"]},
        {"facet_id": "music.now_playing", "app_id": "music", "name": "Now Playing", "capabilities": ["view_title", "pause"]},
        {"facet_id": "recipes.open_section", "app_id": "recipes", "name": "Open Section", "capabilities": ["open_view_recipe"]},
        {"facet_id": "recipes.view_recipe", "app_id": "recipes", "name": "View Recipe", "capabilities": ["view_ingredients", "open_cook_recipe"]},
        {"facet_id": "recipes.cook_recipe", "app_id": "recipes", "name": "Cook Recipe", "capabilities": ["view_step"]},
    ],
    "edges": [
        {"source": "music.open_section", "target": "music.now_playing", "capability": "open_now_playing"},
        {"source": "recipes.open_section", "target": "recipes.view_recipe", "capability": "open_view_recipe"},
        {"source": "recipes.view_recipe", "target": "recipes.cook_recipe", "capability": "open_cook_recipes"},  # typo
    ],
}
events = '''scenario_id,participant_id,task,environment,widget_id,app_id,facet_id,capabilities
s1,u1,cooking,kitchen,1,recipes,recipes.cook_recipe,"{""recipes.view_step"": ""primary""}"
s1,u1,cooking,kitchen,2,music,music.now_playing,"{""music.view_title"": ""ambient""}"
s2,u1,dinner,kitchen,1,recipes,recipes.view_recipe,"{""recipes.view_ingredients"": ""primary""}"
s3,u2,cooking,kitchen,1,recipes,recipes.cook_recipe,"{""recipes.view_step"": ""primary""}"
s4,u2,relaxing,living room,1,music,music.now_playing,"{""music.pause"": ""primary""}"
'''
(root / "facet_db.json").write_text(json.dumps(db))
(root / "events.csv").write_text(events)

In [ ]:
from contextxr.data.validate import validate_dataset

print(validate_dataset(root))

The validator points to the edge whose capability is not on its source facet. Fix it and validate again:

In [ ]:
db["edges"][2]["capability"] = "open_cook_recipe"
(root / "facet_db.json").write_text(json.dumps(db))
print(validate_dataset(root))

## Evaluate

With your own data, pass your own folds (scenario → fold), or `folds=None` to fit on everything. Next facet suggestion places each scenario's targets in sorted order unless you pass `orders=`.

In [ ]:
from IPython.display import Markdown
from contextxr import evaluate
from contextxr.data.loader import load_dataset
from contextxr.methods import GlobalPopularity
from contextxr.runner import format_table

mine = load_dataset(root)
folds = {"s1": 1, "s2": 1, "s3": 2, "s4": 2}
initial = evaluate(GlobalPopularity(), "initial", mine, folds=folds)
nxt = evaluate(GlobalPopularity(), "next", mine, folds=folds)
initial.run["method"]["name"], nxt.run["method"]["name"] = "popularity (initial)", "popularity (next)"
Markdown(format_table([initial, nxt]))

| method | n | manual | cost | reduction | Recall@10 | F1@10 | Hit@10 |
|---|---:|---:|---:|---:|---:|---:|---:|
| popularity (initial) | 4 | 4.25 | 0.62 | -85.3% | 1.000 | 0.220 | 1.000 |
| popularity (next) | 4 | 4.25 | 0.62 | -85.3% | 1.000 | 0.371 | 1.000 |